# LME Copper Data — Cleaning and Exploratory Analysis

This notebook reads the incrementally collected Westmetall data, validates and cleans it, constructs analytical parameters, and explores relationships between copper prices and LME stock. The raw files are never modified.

## 1. Imports and adjustable parameters

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# ---------- Parameters to experiment with ----------
START_DATE = None          # Example: '2015-01-01'; None keeps all observations
END_DATE = None            # Example: '2025-12-31'; None keeps all observations
ROLLING_WINDOWS = [5, 20, 60]  # Approx. week, month, and quarter in trading days
LAG_DAYS = range(-60, 61)  # Negative: stock leads price; positive: price leads stock
MIN_PERIODS_RATIO = 0.75   # Required fraction of observations in rolling windows
WINSORIZE_RETURNS = False  # Turn on only for sensitivity analysis
WINSOR_LIMITS = (0.01, 0.99)
SAVE_PROCESSED_DATA = False  # Change to True after reviewing the transformations

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda value: f'{value:,.4f}')

## 2. Locate and load the raw data

In [ ]:
# This works whether Jupyter starts in the workspace, project, or notebooks directory.
candidate_roots = [Path.cwd(), Path.cwd().parent, Path.cwd() / 'projects' / 'copper']
PROJECT_DIR = next((root for root in candidate_roots if (root / 'data' / 'raw' / 'lme' / 'copper_lme_raw.csv').exists()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError('Could not locate data/raw/lme/copper_lme_raw.csv')

RAW_PATH = PROJECT_DIR / 'data' / 'raw' / 'lme' / 'copper_lme_raw.csv'
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'

raw_df = pd.read_csv(RAW_PATH, dtype='string')
print(f'Raw path: {RAW_PATH.resolve()}')
print(f'Shape: {raw_df.shape}')
raw_df.head()

In [ ]:
raw_df.info()
display(raw_df.tail())
display(raw_df.isna().sum().rename('empty_cells'))
display(raw_df.nunique().rename('unique_values'))

## 3. Clean types and validate

A dash in the source means unavailable, so it becomes `NaN`. Numeric strings retain their original form in `raw_df`; cleaning happens only in `df`.

In [ ]:
df = raw_df.copy()
df['date'] = pd.to_datetime(df['date'], format='%Y-%m-%d', errors='raise')

numeric_columns = ['cash_settlement', 'three_month', 'stock']
for column in numeric_columns:
    df[column] = pd.to_numeric(
        df[column].replace('-', pd.NA).str.replace(',', '', regex=False),
        errors='raise',
    )

df['source_year'] = pd.to_numeric(df['source_year'], errors='raise').astype('int16')
df['fetched_at_utc'] = pd.to_datetime(df['fetched_at_utc'], utc=True, errors='raise')
df = df.sort_values('date').reset_index(drop=True)

if START_DATE is not None:
    df = df.loc[df['date'] >= pd.Timestamp(START_DATE)].copy()
if END_DATE is not None:
    df = df.loc[df['date'] <= pd.Timestamp(END_DATE)].copy()

df.head()

In [ ]:
quality_report = pd.Series({
    'rows': len(df),
    'first_date': df['date'].min(),
    'last_date': df['date'].max(),
    'duplicate_dates': df['date'].duplicated().sum(),
    'weekend_dates': (df['date'].dt.dayofweek >= 5).sum(),
    'missing_cash': df['cash_settlement'].isna().sum(),
    'missing_three_month': df['three_month'].isna().sum(),
    'missing_stock': df['stock'].isna().sum(),
    'source_year_mismatches': (df['date'].dt.year != df['source_year']).sum(),
    'nonpositive_numeric_values': (df[numeric_columns] <= 0).sum().sum(),
})
display(quality_report.to_frame('value'))

assert df['date'].is_monotonic_increasing
assert not df['date'].duplicated().any()
assert (df['date'].dt.year == df['source_year']).all()
assert not (df[numeric_columns] <= 0).any().any()

In [ ]:
# Review exceptional observations instead of silently removing them.
exception_rows = df.loc[
    df[numeric_columns].isna().any(axis=1) | (df['date'].dt.dayofweek >= 5),
    ['date', *numeric_columns, 'source_url'],
]
exception_rows

## 4. Construct analytical parameters

Returns and changes are preferable to raw levels for many correlation tests because price and inventory levels can both trend over time. The spread is cash minus three-month price; a positive value indicates backwardation and a negative value indicates contango.

In [ ]:
df['spread'] = df['cash_settlement'] - df['three_month']
df['spread_pct'] = 100 * df['spread'] / df['three_month']
df['cash_return'] = df['cash_settlement'].pct_change(fill_method=None)
df['cash_log_return'] = np.log(df['cash_settlement']).diff()
df['three_month_return'] = df['three_month'].pct_change(fill_method=None)
df['stock_change'] = df['stock'].diff()
df['stock_pct_change'] = df['stock'].pct_change(fill_method=None)
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day_of_week'] = df['date'].dt.day_name()

for window in ROLLING_WINDOWS:
    minimum = max(2, int(window * MIN_PERIODS_RATIO))
    df[f'cash_ma_{window}d'] = df['cash_settlement'].rolling(window, min_periods=minimum).mean()
    df[f'cash_volatility_{window}d'] = df['cash_log_return'].rolling(window, min_periods=minimum).std() * np.sqrt(252)
    df[f'stock_ma_{window}d'] = df['stock'].rolling(window, min_periods=minimum).mean()

if WINSORIZE_RETURNS:
    for column in ['cash_return', 'cash_log_return', 'three_month_return', 'stock_pct_change']:
        lower, upper = df[column].quantile(WINSOR_LIMITS)
        df[column] = df[column].clip(lower, upper)

df[['date', 'cash_settlement', 'three_month', 'stock', 'spread', 'cash_return', 'stock_pct_change']].tail()

## 5. Distributions and descriptive statistics

In [ ]:
analysis_columns = [
    'cash_settlement', 'three_month', 'stock', 'spread', 'spread_pct',
    'cash_return', 'cash_log_return', 'stock_change', 'stock_pct_change',
]
display(df[analysis_columns].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
sns.histplot(df['cash_settlement'], kde=True, ax=axes[0, 0])
axes[0, 0].set_title('Cash-settlement price distribution')
sns.histplot(df['cash_log_return'].dropna(), bins=80, kde=True, ax=axes[0, 1])
axes[0, 1].set_title('Daily log-return distribution')
sns.histplot(df['stock'], kde=True, ax=axes[1, 0])
axes[1, 0].set_title('LME stock distribution')
sns.histplot(df['spread'].dropna(), bins=80, kde=True, ax=axes[1, 1])
axes[1, 1].axvline(0, color='black', linewidth=1)
axes[1, 1].set_title('Cash minus three-month spread')
plt.tight_layout()

## 6. Time-series behavior

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(15, 12), sharex=True)
axes[0].plot(df['date'], df['cash_settlement'], label='Cash settlement', linewidth=1)
axes[0].plot(df['date'], df['three_month'], label='Three month', linewidth=1, alpha=0.8)
axes[0].set_ylabel('USD per tonne')
axes[0].legend()
axes[0].set_title('LME copper prices')
axes[1].plot(df['date'], df['stock'], color='tab:orange', linewidth=1)
axes[1].set_ylabel('Tonnes')
axes[1].set_title('LME copper stock')
axes[2].plot(df['date'], df['spread'], color='tab:green', linewidth=1)
axes[2].axhline(0, color='black', linewidth=0.8)
axes[2].set_ylabel('USD per tonne')
axes[2].set_title('Cash–three-month spread')
plt.tight_layout()

## 7. Correlations

Pearson measures linear association; Spearman measures monotonic association and is less sensitive to extreme observations. Correlation does not establish causality.

In [ ]:
level_columns = ['cash_settlement', 'three_month', 'stock', 'spread']
change_columns = ['cash_return', 'three_month_return', 'stock_pct_change', 'spread_pct']

pearson_levels = df[level_columns].corr(method='pearson')
spearman_levels = df[level_columns].corr(method='spearman')
pearson_changes = df[change_columns].corr(method='pearson')
spearman_changes = df[change_columns].corr(method='spearman')

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
for matrix, title, axis in [
    (pearson_levels, 'Pearson — levels', axes[0, 0]),
    (spearman_levels, 'Spearman — levels', axes[0, 1]),
    (pearson_changes, 'Pearson — changes', axes[1, 0]),
    (spearman_changes, 'Spearman — changes', axes[1, 1]),
]:
    sns.heatmap(matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, vmin=-1, vmax=1, ax=axis)
    axis.set_title(title)
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.regplot(data=df, x='stock', y='cash_settlement', scatter_kws={'alpha': 0.18, 's': 12}, line_kws={'color': 'red'}, ax=axes[0])
axes[0].set_title('Price level vs stock level')
sns.regplot(data=df, x='stock_pct_change', y='cash_return', scatter_kws={'alpha': 0.18, 's': 12}, line_kws={'color': 'red'}, ax=axes[1])
axes[1].set_title('Daily price return vs stock change')
plt.tight_layout()

## 8. Lagged relationships

This checks whether changes in stock tend to precede or follow price returns. It uses observation lags (trading rows), not exact calendar days.

In [ ]:
lag_correlations = pd.DataFrame({
    'lag': list(LAG_DAYS),
    'pearson': [df['cash_return'].corr(df['stock_pct_change'].shift(lag), method='pearson') for lag in LAG_DAYS],
    'spearman': [df['cash_return'].corr(df['stock_pct_change'].shift(lag), method='spearman') for lag in LAG_DAYS],
})

display(lag_correlations.loc[lag_correlations['pearson'].abs().nlargest(10).index].sort_values('lag'))

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(lag_correlations['lag'], lag_correlations['pearson'], label='Pearson')
ax.plot(lag_correlations['lag'], lag_correlations['spearman'], label='Spearman', alpha=0.8)
ax.axhline(0, color='black', linewidth=0.8)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set(title='Lag correlation: price return vs shifted stock change', xlabel='Stock-change lag (trading observations)', ylabel='Correlation')
ax.legend()
plt.show()

## 9. Correlation stability over time

In [ ]:
yearly_summary = (
    df.groupby('year')
      .agg(
          observations=('date', 'size'),
          average_cash=('cash_settlement', 'mean'),
          year_end_cash=('cash_settlement', 'last'),
          average_stock=('stock', 'mean'),
          average_spread=('spread', 'mean'),
          annualized_volatility=('cash_log_return', lambda x: x.std() * np.sqrt(252)),
      )
)
yearly_summary['price_stock_level_corr'] = df.groupby('year')[['cash_settlement', 'stock']].apply(
    lambda group: group['cash_settlement'].corr(group['stock'])
)
yearly_summary['return_stock_change_corr'] = df.groupby('year')[['cash_return', 'stock_pct_change']].apply(
    lambda group: group['cash_return'].corr(group['stock_pct_change'])
)
display(yearly_summary)

In [ ]:
monthly = (
    df.set_index('date')
      .resample('ME')
      .agg({
          'cash_settlement': 'last',
          'three_month': 'last',
          'stock': 'last',
          'spread': 'mean',
          'cash_log_return': 'std',
      })
)
monthly['cash_monthly_return'] = monthly['cash_settlement'].pct_change(fill_method=None)
monthly['stock_monthly_change'] = monthly['stock'].pct_change(fill_method=None)
monthly['monthly_volatility'] = monthly['cash_log_return'] * np.sqrt(252)

display(monthly.tail(12))
display(monthly[['cash_monthly_return', 'stock_monthly_change', 'spread']].corr())

## 10. Optional processed-data export

Review all transformations first. When satisfied, set `SAVE_PROCESSED_DATA = True` in the parameter cell and rerun.

In [ ]:
if SAVE_PROCESSED_DATA:
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    processed_path = PROCESSED_DIR / 'copper_lme_processed.csv'
    df.to_csv(processed_path, index=False)
    print(f'Saved {len(df):,} rows to {processed_path.resolve()}')
else:
    print('Export skipped. Set SAVE_PROCESSED_DATA = True when ready.')